# Best Local LLMs in Practice (2026 Edition)

This notebook demonstrates how to use the top-performing open-source language models that can run locally with <64GB RAM. We'll focus on the two families this course uses: Gemma 4 (the course default) and Qwen 3.8.

## Prerequisites

- Install Ollama: https://ollama.ai/
- At least 16GB RAM (32GB+ recommended)
- GPU with 8GB+ VRAM (optional but recommended)

## 1. Gemma 4 — The 2026 Course Default

Gemma 4 (Google DeepMind, April 2026) is the **course default** for this session. Pick the size that fits your hardware:
- **`gemma4:e4b`** (9.6 GB) — what bare `gemma4` pulls; CPU-friendly, no GPU required
- **`gemma4:e2b`** (7.2 GB) — smallest variant for low-RAM machines
- **`gemma4:12b`** (≈8 GB download) — targets 16 GB VRAM / unified-memory laptops
- **`gemma4:26b`** (MoE, 19 GB) / **`gemma4:31b`** (20 GB) — for 24 GB GPUs
- **Multimodal** — image input on all sizes; audio on E2B/E4B/12B only (via Hugging Face Transformers, not exposed through Ollama)
- **Apache 2.0** — no MAU caps, no EU restrictions


In [ ]:
import ollama
from IPython.display import Markdown, display

def chat_with_gemma4(prompt, model="gemma4"):
    """Chat with Gemma 4 via Ollama — course default model."""
    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return response['message']['content']
    except Exception as e:
        return f"Error: {e}. Run 'ollama pull {model}' first."

# `gemma4` = gemma4:e4b (CPU-friendly). On a 16 GB laptop try model="gemma4:12b";
# on a 24 GB GPU, model="gemma4:26b" or "gemma4:31b".
response = chat_with_gemma4(
    "Explain in 2 sentences why running LLMs locally matters for privacy."
)
display(Markdown(response))


## 2. Qwen 3.8 27B — Coding + Agentic Tasks

Qwen 3.8 27B (Alibaba, Aug 2026, Apache 2.0, 18 GB) is the newer Qwen release (after Qwen 3.6) and the course's tool-calling alternative to Gemma 4:
- **27B parameters** — `qwen3.8:27b` is an 18 GB download; plan for a 24 GB GPU (or slow CPU offload)
- **Tool calling + coding** — tuned for coding and long-horizon agentic tasks
- **Thinking mode** — on by default, can be disabled per request
- **256K context window**, text + image input


In [ ]:
def chat_with_qwen(prompt, model="qwen3.8:27b"):
    """Chat with Qwen 3.8 27B via Ollama — alternative for agentic/coding tasks."""
    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return response['message']['content']
    except Exception as e:
        return f"Error: {e}. Run 'ollama pull {model}' first."

response = chat_with_qwen(
    "Write a Python function that retries a failing HTTP request with exponential backoff."
)
display(Markdown(response))


## 3. Model Comparison - Same Task, Different Strengths

Let's give the same complex task to different models to see their unique approaches:

In [ ]:
comparison_prompt = """
Design a distributed system for real-time fraud detection that can handle:
- 100,000 transactions per second
- Sub-100ms latency requirement
- 99.99% availability
- Global deployment

Provide:
1. High-level architecture diagram (in text/ASCII)
2. Technology stack recommendations
3. Scaling strategy
4. Key challenges and solutions

Keep it practical and implementable.
"""

print("=== GEMMA 4 RESPONSE ===")
gemma_response = chat_with_gemma4(comparison_prompt)
display(Markdown(gemma_response))

print("\n\n=== QWEN 3.8 RESPONSE ===")
qwen_response = chat_with_qwen(comparison_prompt)
display(Markdown(qwen_response))

## 4. Performance Monitoring and Resource Usage

Let's monitor the performance characteristics of different models:

In [ ]:
import time
import psutil
import matplotlib.pyplot as plt

def benchmark_model(model_name, prompt, chat_function):
    """Benchmark model performance"""
    # Get initial memory usage
    initial_memory = psutil.virtual_memory().used / (1024**3)  # GB
    
    # Time the response
    start_time = time.time()
    response = chat_function(prompt)
    end_time = time.time()
    
    # Get final memory usage
    final_memory = psutil.virtual_memory().used / (1024**3)  # GB
    
    # Calculate metrics
    response_time = end_time - start_time
    memory_used = final_memory - initial_memory
    tokens_estimated = len(response.split())  # Rough estimate
    tokens_per_second = tokens_estimated / response_time if response_time > 0 else 0
    
    return {
        'model': model_name,
        'response_time': response_time,
        'memory_used': memory_used,
        'tokens_estimated': tokens_estimated,
        'tokens_per_second': tokens_per_second,
        'response_length': len(response)
    }

# Simple benchmark prompt
benchmark_prompt = "Explain the concept of machine learning in exactly 100 words."

# Benchmark the two course models
results = []

results.append(benchmark_model("Gemma 4 (gemma4)", benchmark_prompt, chat_with_gemma4))
results.append(benchmark_model("Qwen 3.8 27B", benchmark_prompt, chat_with_qwen))

# Display results
if results:
    for result in results:
        print(f"\n{result['model']} Performance:")
        print(f"  Response Time: {result['response_time']:.2f}s")
        print(f"  Estimated Tokens/sec: {result['tokens_per_second']:.1f}")
        print(f"  Response Length: {result['response_length']} chars")
else:
    print("No benchmark results collected")

## 5. Advanced Use Cases

### Tool Calling and Function Integration

In [ ]:
# Example of using models for tool calling/function generation
tool_calling_prompt = """
Create a Python class that acts as a "Smart Assistant" with the following capabilities:

1. Weather lookup (mock API call)
2. Calendar management (add/remove events)
3. Email composition
4. Web search (mock implementation)
5. File operations (create, read, update files)

The class should:
- Have a unified interface for natural language commands
- Parse user intent and route to appropriate functions
- Handle errors gracefully
- Return structured responses

Include example usage showing how to:
- "What's the weather in New York?"
- "Schedule a meeting for tomorrow at 2 PM"
- "Write an email to John about the project update"
"""

response = chat_with_qwen(tool_calling_prompt)
display(Markdown(response))

### RAG (Retrieval-Augmented Generation) Integration

In [ ]:
rag_prompt = """
Design and implement a RAG (Retrieval-Augmented Generation) system that:

1. Ingests documents from multiple formats (PDF, Word, HTML, Markdown)
2. Creates embeddings using sentence-transformers
3. Stores embeddings in a vector database (Chroma or FAISS)
4. Implements semantic search with relevance scoring
5. Combines retrieved context with LLM generation

Requirements:
- Handle documents up to 10MB
- Support real-time updates
- Implement caching for performance
- Include source attribution

Provide complete Python implementation with:
- Document processing pipeline
- Vector storage and retrieval
- Integration with local LLM
- Example usage
"""

response = chat_with_gemma4(rag_prompt)
display(Markdown(response))

For which model and size to pick on your hardware, see the [2026 local model guide](../best-local-models-2026.md).

## Conclusion

This notebook demonstrated the practical capabilities of the two model families this course uses:

- **Gemma 4**: The course default, from `gemma4:e4b` on CPU to `gemma4:31b` on a 24 GB GPU
- **Qwen 3.8 27B**: The tool-calling and coding alternative

Each model has unique strengths, and the choice depends on your specific use case, available hardware, and performance requirements.

For more detailed information, see the accompanying [`best-local-models-2026.md`](../best-local-models-2026.md) guide.